# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


### Tools

In [ ]:
import importlib, pkgutil   # 모듈을 동적 로드 / 패키지 탐색 유틸

# langchain_community.tools 패키지 로드
package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name)  # 각 모듈(도구) 이름 출력

# Wikipedia Tool

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki_tool = WikipediaQueryRun(api_wrapper= WikipediaAPIWrapper())
# print(wiki_tool.run('Physical AI'))

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '')]

llm = init_chat_model('gpt-5.6-luna')
print(llm.invoke('걸그룹 튜이드 멤버 알려줘'))

### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [ ]:
import requests
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

@tool
def search_arxiv(arxiv_id: str) -> str:
    """arxiv 논문 ID로 제목, 저자, 초록을 조회합니다."""
    
    url = "https://export.arxiv.org/api/query"
    response = requests.get(
        url,
        params = {
            "id_list": arxiv_id,
            "max_results": 1
        },
        timeout = 10
    )
    
    response.raise_for_status()
    
    root = ET.fromstring(response.text)
    
    ns = {'atom': "http://www.w3.org/2005/Atom"}
    entry = root.find('atom:entry', ns)
    
    if entry is None:
        return '논문 정보를 찾을 수 없습니다.'
    
    title = entry.findtext('atom:title', namespaces=ns).strip()
    summary = entry.findtext('atom:summary', namespaces=ns).strip()
    authors = [author.findtext('atom:name', namespaces=ns)
               for author in entry.findall('atom:author', ns)]
    
    return f"""
제목 : {title}
저자 : {','.join(authors)}
초록 : {summary}
"""

In [ ]:
tools = [search_arxiv, wiki_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해주세요.'
)

messages = [('human', '1706.03762 이 논문의 내용을 간단하게 설명해줄래? (한글답변)' )]
response = agent.invoke({'messages': messages})

print(response['messages'][-1].content)

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools
llm = init_chat_model('gpt-4.1-mini')
tools =load_tools( ['wikipedia', 'llm-math'], llm = llm)


agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt='''
    당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변을 작성해주세요.
    단, 숫자계산은 반드시 llm-math 도구를 사용하여 답변에 활용해야 합니다.
    '''
)

response = agent.invoke({'messages': '3.5의 3제곱은 몇이야? 그리고 그결과에 5를 더해줘'})

print(response['messages'][-1].content)

### duckduckgo
https://docs.langchain.com/oss/python/integrations/tools/ddg

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

### duckduckgo
https://docs.langchain.com/oss/python/integrations/tools/ddg

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results = 3,
    topic = 'general',
    include_images = True,
    search_depth = 'advanced'
)


tavily_tool.invoke('2026년 8월 현재 대한민국에서 가장 핫한 이슈는 뭐야?')

# @tool


In [ ]:
# eval / exec 로 문자열 코드 실행
a= 10
print(eval('5 + 3 + a'))
exec('b= 10')
print(b)

In [ ]:
from langchain_core.tools import tool

@tool
def simple_calculator(query:str) -> str:
    """
    산술연산을 위한 간단한 계산기 Tool
    Args:
        query: 계산식
    Return:
        계산식 결과값
        
    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4**2 / 8") -> "계산 결과: 2"
    """
    try:
        result = eval(query)
        return f"계산결과: {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"
    
simple_calculator

In [ ]:
llm = init_chat_model('gpt-5.4-mini')

tools =[simple_calculator]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해주세요.'
)

response = agent.invoke({'messages': "7 + 3 * 8 이거를 계산해줘."})

pprint(response)
print('='*50)
pprint(response['messages'][-1].content)

In [ ]:
import json
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

@tool
def get_current_weather(city= 'Seoul', units= 'metric'):
    """
        OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

        Args:
            - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
                - 변환예시:
                    - 서울 -> Seoul
                    - 충남, 충청남도 -> Chungcheongnam-do
                    - 부산 -> Busan
            - units: str 온도단위를 설정하는 문자열
              - metric(기본값: 섭씨, 미터)
              - imperial(화씨, 야드)
        Return:
            - str: json 형식으로 변환된 현재 날씨 정보
    """
    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()
    
    weather_info= {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp']
        temp_feels_like = data['main']['feels_like']
        humidity = data['main']['humidity']

        weather_info = {
            'city': city,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }
    else:
        weather_info = {
            'city': city,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }
    
    return json.dumps(weather_info)

get_current_weather

In [ ]:
llm = init_chat_model('gpt-5.4-mini')

tools =[simple_calculator, get_current_weather]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해주세요.'
)

response = agent.invoke({'messages': "나오늘 뭐입어야돼? 강원도살어"})

pprint(response)
print('='*50)
pprint(response['messages'][-1].content)

In [ ]:
# 한국 기준 현재 날짜/ 시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone

@tool
def get_current_datetime(format: str= '%Y-%m-%d %H:%M:%S')->str:
    """
    한국기준 현재시각정보를 반환하는 함수
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """
    kst = timezone('Asia/Seoul')
    return datetime.now(kst).strftime(format)

get_current_datetime

In [ ]:
@tool
def calculate_age(today_date: str, birth_date: str)-> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """
    try:
        today = datetime.strptime(today_date, '%Y-%m-%d')
        birthday = datetime.strptime(birth_date,'%Y-%m-%d' )
        
        age = today.year - birthday.year
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -= 1
        return age
    except ValueError:
        return '날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해 주세요.'
    
calculate_age.invoke({'today_date': '2026-08-27', 'birth_date': '1964-02-14'})

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

llm = init_chat_model('gpt-5.4-mini')

tools = [TavilySearch()]

agent = create_agent(llm, tools, checkpointer= InMemorySaver())

response = agent.invoke(
    input = {'messages': [('human', '안녕! 만나서 반갑다! 나는 cap이라고 해. 넌 누구니?')]},
    config = {'configurable': {'thread_id': '100'}}
)

print(response['messages'][-1].content)

# sqliteSaver

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()        # 테이블 생성 밒 초기화 (기존에 존재하면 그대로 사용)
    
    agent = create_agent(llm, tools, checkpointer= checkpointer)

    response = agent.invoke(
    input = {'messages': [('human', '오케이 완전 이해했어! 그럼너가 말해준 langchain, langgraph를 사용해볼게 ')]},
    config = {'configurable': {'thread_id': '100'}}
)
    


pprint(response)
print('='*50)
pprint(response['messages'][-1].content)

In [ ]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({'configurable': {'thread_id': '100'}})
    
    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']
    
    for i, message in enumerate(messages, 1):
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f'{i}: [{msg_type}] {message.content}')
        print()